# Data Loading — formative2-hmm

This notebook covers the full data loading pipeline:
1. **Configuration** — constants (target Hz, trim duration, split ratio)
2. **Helper functions** — sensor reading, resampling, merging, trimming
3. **Discover & process** — unpack ZIPs, clean, assign train/test split, write CSVs
4. **Verification** — inspect one cleaned file and print a dataset summary

## 1. Configuration

In [1]:
# ── Configuration constants ────────────────────────────────────────────────────
TARGET_HZ      = 100       # resample every recording to this rate (Hz)
EDGE_TRIM_SEC  = 2.0       # seconds to remove from both ends of each recording
MERGE_TOL_SEC  = 0.01      # max time gap (s) when aligning acc ↔ gyro
RANDOM_SEED    = 42
TRAIN_RATIO    = 0.8       # fraction of recordings used for training

print(f"TARGET_HZ={TARGET_HZ}, EDGE_TRIM_SEC={EDGE_TRIM_SEC}, "
      f"TRAIN_RATIO={TRAIN_RATIO}")

TARGET_HZ=100, EDGE_TRIM_SEC=2.0, TRAIN_RATIO=0.8


## 2. Imports & Path Setup

In [2]:
import io
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Paths relative to this notebook (notebooks/ → project root → data/)
PROJECT_ROOT = Path("..").resolve()
RAW_DIR      = PROJECT_ROOT / "data"
OUT_DIR      = PROJECT_ROOT / "data" / "processed"

print("Project root :", PROJECT_ROOT)
print("Raw data dir :", RAW_DIR)
print("Output dir   :", OUT_DIR)

Project root : /home/damour/Documents/ALU/formative2-hmm
Raw data dir : /home/damour/Documents/ALU/formative2-hmm/data
Output dir   : /home/damour/Documents/ALU/formative2-hmm/data/processed


## 3. Helper Functions

### 3a. Read a single sensor (accelerometer or gyroscope) from a ZIP

In [9]:
def read_sensor(zp: zipfile.ZipFile, keywords: list) -> pd.DataFrame | None:
    """
    Return the first CSV inside *zp* whose filename contains one of *keywords*.
    Output columns: [time_s, x, y, z] with time normalised to start at 0 s.
    Returns None if no matching file is found.
    """
    target = None
    for info in zp.infolist():
        nm = info.filename.lower()
        if nm.endswith(".csv") and any(k in nm for k in keywords):
            target = info
            break
    if target is None:
        return None

    with zp.open(target) as f:
        df = pd.read_csv(io.BytesIO(f.read()))

    # ── locate the time column ─────────────────────────────────────────────────
    lower = {c.lower(): c for c in df.columns}
    for cand in ("seconds_elapsed", "timestamp", "time", "epoch(s)", "epoch(ms)"):
        if cand in lower:
            tcol = lower[cand]
            break
    else:
        tcol = df.columns[0]

    t = df[tcol].astype(float).to_numpy().copy()  # writable copy
    if t.max() > 1e6:          # milliseconds → seconds
        t = t / 1000.0
    t = t - t[0]               # normalise to 0

    # ── locate x/y/z columns ──────────────────────────────────────────────────
    def pick(*names):
        for n in names:
            if n in df:
                return df[n].astype(float).to_numpy()
            if n.upper() in df:
                return df[n.upper()].astype(float).to_numpy()
        return None

    x = pick("x", "ax", "accx")
    y = pick("y", "ay", "accy")
    z = pick("z", "az", "accz")

    if x is None or y is None or z is None:
        others = [c for c in df.columns
                  if c != tcol and pd.api.types.is_numeric_dtype(df[c])]
        if len(others) >= 3:
            x, y, z = (df[others[i]].to_numpy(float) for i in range(3))
        else:
            raise ValueError("x/y/z axes not found")

    out = pd.DataFrame({"time_s": t, "x": x, "y": y, "z": z})
    return (out.drop_duplicates(subset="time_s")
               .sort_values("time_s")
               .reset_index(drop=True))

print("read_sensor() defined ✓")

read_sensor() defined ✓


### 3b. Resample to a fixed frequency

In [10]:
def estimate_hz(t: np.ndarray) -> float:
    """Estimate sampling rate (Hz) from a time array via median Δt."""
    if len(t) < 2:
        return float("nan")
    dt = np.diff(t)
    dt = dt[(dt > 0) & np.isfinite(dt)]
    return float(1.0 / np.median(dt)) if len(dt) else float("nan")


def resample(df: pd.DataFrame, hz: int) -> pd.DataFrame:
    """Linear interpolation to a uniform *hz* Hz time grid."""
    if df is None or df.empty:
        return df
    t0, t1 = df["time_s"].iloc[0], df["time_s"].iloc[-1]
    new_t = np.arange(t0, t1 + 1e-9, 1.0 / hz)
    out = pd.DataFrame({"time_s": new_t})
    for c in ("x", "y", "z"):
        out[c] = np.interp(new_t, df["time_s"].to_numpy(), df[c].to_numpy())
    return out

print("estimate_hz() and resample() defined ✓")

estimate_hz() and resample() defined ✓


### 3c. Merge accelerometer + gyroscope on time axis

In [11]:
def merge_sensors(
    acc: pd.DataFrame | None,
    gyr: pd.DataFrame | None,
    tol: float = MERGE_TOL_SEC,
) -> pd.DataFrame:
    """
    Nearest-time merge of accelerometer and gyroscope DataFrames.
    Output columns: [time_s, ax, ay, az, gx, gy, gz]
    Missing sensor is filled with zeros.
    """
    COLS = ["time_s", "ax", "ay", "az", "gx", "gy", "gz"]
    if acc is None and gyr is None:
        return pd.DataFrame(columns=COLS)
    if acc is None:
        g = gyr.rename(columns={"x": "gx", "y": "gy", "z": "gz"}).copy()
        g[["ax", "ay", "az"]] = 0.0
        return g[COLS]
    if gyr is None:
        a = acc.rename(columns={"x": "ax", "y": "ay", "z": "az"}).copy()
        a[["gx", "gy", "gz"]] = 0.0
        return a[COLS]

    A = acc.sort_values("time_s")
    G = (gyr.sort_values("time_s")
             .rename(columns={"x": "gx", "y": "gy", "z": "gz"}))
    m = pd.merge_asof(A, G, on="time_s", tolerance=tol, direction="nearest")
    m = m.dropna(subset=["gx", "gy", "gz"])
    return (m.rename(columns={"x": "ax", "y": "ay", "z": "az"})[COLS]
             .reset_index(drop=True))

print("merge_sensors() defined ✓")

merge_sensors() defined ✓


### 3d. Trim edges & assign train/test split

In [12]:
def trim_edges(df: pd.DataFrame, sec: float = EDGE_TRIM_SEC) -> pd.DataFrame:
    """Remove the first and last *sec* seconds from a recording."""
    if df.empty:
        return df
    t0 = df["time_s"].iloc[0] + sec
    t1 = df["time_s"].iloc[-1] - sec
    if t1 <= t0:
        return df.iloc[0:0].copy()
    return df[(df["time_s"] >= t0) & (df["time_s"] <= t1)].reset_index(drop=True)


REC_ID_RE = re.compile(r"(\d+)\.zip$", re.IGNORECASE)

def assign_split(activity: str, rec_id: int,
                 train_ratio: float = TRAIN_RATIO,
                 seed: int = RANDOM_SEED) -> str:
    """Deterministic train/test label derived from (activity, rec_id)."""
    h = hash((activity, rec_id, seed)) & 0x7FFFFFFF
    return "train" if (h / 0x7FFFFFFF) < train_ratio else "test"

print("trim_edges() and assign_split() defined ✓")

trim_edges() and assign_split() defined ✓


## 4. Discover Recordings

Scan every activity sub-folder under `data/` and collect all ZIP files.

In [13]:
def discover_recordings(data_dir: Path) -> list:
    """
    Walk data_dir/<activity>/*.zip and return
    [(zip_path, activity, rec_id), ...] sorted by activity then rec_id.
    """
    recordings = []
    for sub in sorted(data_dir.iterdir()):
        if not sub.is_dir():
            continue
        activity = sub.name.lower()
        for zpath in sorted(sub.glob("*.zip")):
            m = REC_ID_RE.search(zpath.name)
            if not m:
                raise ValueError(f"Cannot parse rec_id from: {zpath.name}")
            recordings.append((zpath, activity, int(m.group(1))))
    return recordings


recordings = discover_recordings(RAW_DIR)
print(f"Found {len(recordings)} recordings:")
for zp, act, rid in recordings:
    print(f"  {act:10s}  rec={rid:>2d}  {zp.name}")

Found 50 recordings:
  jumping     rec= 3  2026-03-05_18-40-03.zip
  jumping     rec=35  2026-03-05_18-40-35.zip
  jumping     rec=54  2026-03-05_18-40-54.zip
  jumping     rec=13  2026-03-05_18-41-13.zip
  jumping     rec=31  2026-03-05_18-41-31.zip
  jumping     rec= 4  2026-03-05_18-42-04.zip
  jumping     rec= 8  2026-03-05_18-43-08.zip
  jumping     rec=27  2026-03-05_18-43-27.zip
  jumping     rec=47  2026-03-05_18-43-47.zip
  jumping     rec= 4  2026-03-05_18-44-04.zip
  jumping     rec=12  2026-03-05_19-05-12.zip
  jumping     rec=31  2026-03-05_19-05-31.zip
  standing    rec= 1  standing1.zip
  standing    rec=10  standing10.zip
  standing    rec=11  standing11.zip
  standing    rec=12  standing12.zip
  standing    rec= 2  standing2.zip
  standing    rec= 3  standing3.zip
  standing    rec= 4  standing4.zip
  standing    rec= 5  standing5.zip
  standing    rec= 6  standing6.zip
  standing    rec= 7  standing7.zip
  standing    rec= 8  standing8.zip
  standing    rec= 9  standi

## 5. Process & Save Cleaned CSVs

Unpack every ZIP, resample to `TARGET_HZ`, merge acc+gyro, trim edges, assign split, and write to `data/processed/train/` or `data/processed/test/`.

In [14]:
(OUT_DIR / "train").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "test").mkdir(parents=True, exist_ok=True)

saved = []
for zpath, activity, rec_id in recordings:
    split = assign_split(activity, rec_id)

    with zipfile.ZipFile(zpath, "r") as zp:
        acc = read_sensor(zp, ["acc", "accelerometer"])
        gyr = read_sensor(zp, ["gyro", "gyroscope"])

    # resample each sensor independently if needed
    for sensor in (acc, gyr):
        if sensor is not None:
            hz = estimate_hz(sensor["time_s"].to_numpy())
            if np.isfinite(hz) and abs(hz - TARGET_HZ) > 1.0:
                sensor = resample(sensor, TARGET_HZ)

    merged = merge_sensors(acc, gyr)
    merged = trim_edges(merged)

    merged["activity"]     = activity
    merged["split"]        = split
    merged["recording_id"] = rec_id

    dest = OUT_DIR / split / f"{zpath.stem}_cleaned.csv"
    merged.to_csv(dest, index=False)
    saved.append({"activity": activity, "rec_id": rec_id,
                  "split": split, "rows": len(merged), "file": dest.name})
    print(f"  ✓  {dest.relative_to(PROJECT_ROOT)}  ({len(merged)} rows)")

print(f"\nDone — {len(saved)} files written.")

  ✓  data/processed/train/2026-03-05_18-40-03_cleaned.csv  (321 rows)
  ✓  data/processed/test/2026-03-05_18-40-35_cleaned.csv  (331 rows)
  ✓  data/processed/train/2026-03-05_18-40-54_cleaned.csv  (325 rows)
  ✓  data/processed/train/2026-03-05_18-41-13_cleaned.csv  (324 rows)
  ✓  data/processed/train/2026-03-05_18-41-31_cleaned.csv  (304 rows)
  ✓  data/processed/test/2026-03-05_18-42-04_cleaned.csv  (323 rows)
  ✓  data/processed/train/2026-03-05_18-43-08_cleaned.csv  (343 rows)
  ✓  data/processed/train/2026-03-05_18-43-27_cleaned.csv  (318 rows)
  ✓  data/processed/train/2026-03-05_18-43-47_cleaned.csv  (342 rows)
  ✓  data/processed/test/2026-03-05_18-44-04_cleaned.csv  (311 rows)
  ✓  data/processed/train/2026-03-05_19-05-12_cleaned.csv  (374 rows)
  ✓  data/processed/train/2026-03-05_19-05-31_cleaned.csv  (341 rows)
  ✓  data/processed/train/standing1_cleaned.csv  (363 rows)
  ✓  data/processed/test/standing10_cleaned.csv  (344 rows)
  ✓  data/processed/train/standing11_cleane

## 6. Verification — Dataset Summary

In [15]:
summary_df = pd.DataFrame(saved)
print("=== Recordings per activity & split ===")
print(summary_df.groupby(["activity", "split"]).size().unstack(fill_value=0))
print()
print("=== Total rows per activity ===")
print(summary_df.groupby("activity")["rows"].sum())

=== Recordings per activity & split ===
split     test  train
activity             
jumping      3      9
standing     3      9
still        5      8
walking      2     11

=== Total rows per activity ===
activity
jumping     3957
standing    4170
still       4285
walking     4609
Name: rows, dtype: int64


In [16]:
# ── Peek at one cleaned file ───────────────────────────────────────────────────
sample_file = next((OUT_DIR / "train").glob("*.csv"), None)
if sample_file:
    sample = pd.read_csv(sample_file)
    print(f"Sample file : {sample_file.name}")
    print(f"Shape       : {sample.shape}")
    print(f"Columns     : {list(sample.columns)}")
    display(sample.head())

Sample file : walking3_cleaned.csv
Shape       : (356, 10)
Columns     : ['time_s', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'activity', 'split', 'recording_id']


,time_s,ax,ay,az,gx,gy,gz,activity,split,recording_id
0,2.007491,-0.096160,-0.464916,-1.275585,0.107068,0.070762,0.070313,walking,train,3
1,2.027591,-0.337566,-0.468229,-1.251572,0.221081,-0.044749,0.067032,walking,train,3
2,2.047641,-0.357419,-0.600447,-0.736800,0.197149,-0.136462,0.050162,walking,train,3
3,2.067742,-0.112711,-0.525914,-0.082661,0.065417,-0.181394,0.049579,walking,train,3
4,2.087817,0.204625,-0.473762,0.476483,0.028861,-0.052460,0.062369,walking,train,3
